## Testing the pipeline
This notebook aims to test how reliable the knime2galaxy pipeline yields results.

### 1. Test the same knime sample multiple times to check how often the translation is successfull

In [1]:
from translate import translate_knime_to_galaxy

In [2]:
for i in range(5):
    workflow = translate_knime_to_galaxy(
        knwf_path="data/file_to_translate/2025_03_2D_spot_detection.knwf",
        tools_metadata_path="data/tools_metadata.json",
        translation_table_path="data/translation_table.yml",
        workflow_examples_yml_path="data/workflow_translation_table.yml",
        output_galaxy_workflow_path=f"data/output_file/test_i={i}_knime2galaxy_output.ga")

[[{'description': 'Input Image dataset', 'Galaxy': '"0": {\n        "annotation": "",\n        "content_id": null,\n        "errors": null,\n        "id": 0,\n        "input_connections": {},\n        "inputs": [\n            {\n                "description": "",\n                "name": "Input Image Dataset"\n            }\n        ],\n        "label": "Input Image Dataset",\n        "name": "Input dataset",\n        "outputs": [],\n        "position": {\n            "left": 0.0,\n            "top": 0.0\n        },\n        "tool_id": null,\n        "tool_state": "{\\"optional\\": false, \\"tag\\": null}",\n        "tool_version": null,\n        "type": "data_input",\n        "uuid": "312178b3-3fac-49ab-95cf-29e3f44e6015",\n        "when": null,\n        "workflow_outputs": []\n    },\n', 'Python': "from skimage import io\n\ninput_image = io.imread('input_image.tiff')\n", 'KNIME': '<?xml version="1.0" encoding="UTF-8"?>\n<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi

### 2. Test several samples to check how the pipeline handles other examples
For this, the workflows from the __additional_workflows__ branch are used and integrated into the current branch. They are stored under [Galaxy](simple_workflow/Galaxy.tar.xz) and [Knime](simple_workflow/KNIME.tar.xz) respectively.

In [7]:
# extract the tar files
import tarfile
import os

tar_path = "../simple_workflow/Galaxy.tar.xz"
galaxy_path = "../Galaxy_WFs"

# Create output folder if it doesn't exist
os.makedirs(galaxy_path, exist_ok=True)

with tarfile.open(tar_path, "r:xz") as tar:
    tar.extractall(path=galaxy_path)

print("Extraction complete.")

Extraction complete.


In [8]:
tar_path = "../simple_workflow/KNIME.tar.xz"
knime_path = "../Knime_WFs"

# Create output folder if it doesn't exist
os.makedirs(knime_path, exist_ok=True)

with tarfile.open(tar_path, "r:xz") as tar:
    tar.extractall(path=knime_path)

print("Extraction complete.")

Extraction complete.


In [2]:
import re

# 1. sending to the knime2galaxy pipeline
files = os.listdir(knime_path)
regex_names = []

for f in files:
    name = re.sub(r'^\d{4}_\d{2}_|\.knwf$', '', f)
    regex_names.append(name)
    print(name)

resize_rotate_nested
image_conversion
resize_rotate
segmentation_morph_ope_nested
2D_spot_detection
segmentation_morph_ope
image_conversion_nested


In [3]:
finished_files = os.listdir("data/output_file/planemo_test/")
for i,file in enumerate(files):
    filename = name = re.sub(r'^\d{4}_\d{2}_|\.knwf$', '', file)
    filename_ga = filename + "_v2.ga"
    if filename_ga in finished_files:
        print(f"File {file} already processed - skipping")
        continue 
    else:
        workflow = translate_knime_to_galaxy(
            knwf_path= f"../Knime_WFs/{file}",
            tools_metadata_path="data/tools_metadata.json",
            translation_table_path="data/translation_table.yml",
            workflow_examples_yml_path="data/workflow_translation_table.yml",
            output_galaxy_workflow_path=f"data/output_file/planemo_test/{regex_names[i]}_v2.ga")

[[{'description': 'Input Image dataset', 'Galaxy': '"0": {\n        "annotation": "",\n        "content_id": null,\n        "errors": null,\n        "id": 0,\n        "input_connections": {},\n        "inputs": [\n            {\n                "description": "",\n                "name": "Input Image Dataset"\n            }\n        ],\n        "label": "Input Image Dataset",\n        "name": "Input dataset",\n        "outputs": [],\n        "position": {\n            "left": 0.0,\n            "top": 0.0\n        },\n        "tool_id": null,\n        "tool_state": "{\\"optional\\": false, \\"tag\\": null}",\n        "tool_version": null,\n        "type": "data_input",\n        "uuid": "312178b3-3fac-49ab-95cf-29e3f44e6015",\n        "when": null,\n        "workflow_outputs": []\n    },\n', 'Python': "from skimage import io\n\ninput_image = io.imread('input_image.tiff')\n", 'KNIME': '<?xml version="1.0" encoding="UTF-8"?>\n<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi

In [5]:
# 2. use worklfow_lint to check for valid .ga file structure
import subprocess

results = []
outputs = os.listdir("data/output_file/planemo_test")
valid_results = 0

for output in outputs:
    if output.startswith("."):
        continue
    result = subprocess.run(
        ["planemo", "workflow_lint", f"data/output_file/planemo_test/{output}"],
        capture_output=True,
        text=True
    )
    stdout = result.stdout
    stderr = result.stderr
    has_error = "ERROR" in stdout or "ERROR" in stderr

    if has_error:
        print("Lint errors present for workflow", output)
        print(stdout)
    else:
        print("Lint passed for workflow", output)
        valid_results += 1

    results.append(result)
print(f"Valid results: {valid_results} out of {len(files)*2}")

Lint passed for workflow resize_rotate_v2.ga
Lint passed for workflow resize_rotate_nested_v2.ga
Lint passed for workflow segmentation_morph_ope_nested.ga
Lint errors present for workflow image_conversion_v2.ga
.. WARNING: Creator identifier "0000-0003-2104-9519" should be a fully qualified URI, for example "https://orcid.org/0000-0002-1825-0097".
.. WARNING: Workflow step with ID 0 has no annotation.
.. WARNING: Workflow step with ID 0 has no label.
.. WARNING: Workflow step with ID 1 has no annotation.
.. WARNING: Workflow step with ID 1 has no label.
.. WARNING: Workflow missing test cases.
.. ERROR: The ToolShed returned an error when searching for the most recent version of toolshed.g2.bx.psu.edu/repos/imgteam/img_writer/ip_image_writer/0.1+galaxy0: list index out of range

Lint passed for workflow image_conversion_nested_v2.ga
Lint passed for workflow 2D_spot_detection.ga
Lint passed for workflow segmentation_morph_ope_nested_v2.ga
Lint passed for workflow 2D_spot_detection_v2.ga

In [12]:
# check whether the error tools are part of the prompting data or whether it is hallucinated by the model
error_tool_1 = "toolshed.g2.bx.psu.edu/repos/imgteam/img_writer/ip_image_writer/0.1+galaxy0"
error_tool_2 = "toolshed.g2.bx.psu.edu/repos/imgteam/median_filter/ip_median_filter/0.1+galaxy0"
error_tool_3 = "toolshed.g2.bx.psu.edu/repos/imgteam/binary_morphology/ip_binary_morphology/0.1+galaxy0"
correct_tool = "toolshed.g2.bx.psu.edu/repos/imgteam/bfconvert/ip_convertimage/6.7.0+galaxy3"
import json

with open("data/tools_metadata.json") as f:
    data = json.load(f)

def search_values(obj, search_term):
    if isinstance(obj, dict):
        for v in obj.values():
            if search_values(v, search_term):
                return True
    elif isinstance(obj, list):
        for item in obj:
            if search_values(item, search_term):
                return True
    elif isinstance(obj, str):
        if search_term in obj:
            return True
    return False

print(search_values(data, error_tool_1))
print(search_values(data, error_tool_2))
print(search_values(data, error_tool_3))
print(search_values(data, correct_tool)) # just some sort of test whether it finds correctly working tools (that are in the non-erroneous version)

False
False
False
True


## Results
Results suggest that the incorrect tools (those that lead to Errors) are not given to the model, but are hallucinated by it.